# 05 — The Underwriting Report: Putting It All Together

Generate a complete PDF assessment report showing how all components come together in an underwriting workflow. We show two cases: a **clean case** and a **flagged case**.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from medrisk.data.synthetic import generate_cohort, cohort_to_dataframe
from medrisk.features.engineering import build_feature_matrix, get_targets
from medrisk.models.xgb_classifier import RiskClassifier, RiskClassifierConfig
from medrisk.models.cox_ph import CoxPHModel
from medrisk.validation.data_quality import compute_dqs
from medrisk.validation.failure_detection import detect_failure_modes
from medrisk.reporting.pdf_report import ReportData, generate_report

%matplotlib inline

In [2]:
cohort = generate_cohort(n_per_market=500, seed=42)
df = cohort_to_dataframe(cohort)
X, feature_names = build_feature_matrix(df)
events, times = get_targets(df)

# Train models
clf = RiskClassifier(RiskClassifierConfig(n_estimators=100, max_depth=4))
clf.fit(X, events)

cox_features = ["age", "bmi", "sex_male", "charlson_index", "smoking_current"]
cox_df = X.copy()
cox_df["time_to_event"] = df["time_to_event"].values
cox_df["event_occurred"] = df["event_occurred"].values
cox = CoxPHModel(penalizer=0.01)
cox.fit(cox_df, feature_cols=cox_features)

# DQS for all patients
dqs_results = [compute_dqs(p) for p in cohort]
y_proba = clf.predict_proba(X)

print("Models trained. Ready to generate reports.")

Models trained. Ready to generate reports.


## Selecting representative cases

In [3]:
# Clean case: DE market, high DQS
de_indices = [i for i, p in enumerate(cohort) if p.market.value == "DE"]
de_dqs = [(i, dqs_results[i].dqs) for i in de_indices]
clean_idx = max(de_dqs, key=lambda x: x[1])[0]

# Flagged case: INT market, low DQS, high confidence
int_indices = [i for i, p in enumerate(cohort) if p.market.value == "INT"]
int_candidates = [(i, dqs_results[i].dqs, max(y_proba[i], 1-y_proba[i]))
                  for i in int_indices if dqs_results[i].dqs < 0.60]
if int_candidates:
    flagged_idx = max(int_candidates, key=lambda x: x[2])[0]
else:
    flagged_idx = int_indices[0]

print(f"Clean case: index {clean_idx}, DQS={dqs_results[clean_idx].dqs:.2f}, market=DE")
print(f"Flagged case: index {flagged_idx}, DQS={dqs_results[flagged_idx].dqs:.2f}, market=INT")

Clean case: index 414, DQS=0.79, market=DE
Flagged case: index 1892, DQS=0.58, market=INT


## Report 1: A well-documented patient

In [4]:
p = cohort[clean_idx]
dqs_r = dqs_results[clean_idx]
proba = float(y_proba[clean_idx])

val = detect_failure_modes(
    patient_id=p.patient_id, dqs=dqs_r.dqs, dqs_tier=dqs_r.tier,
    raw_proba=proba,
)

data = ReportData(
    patient_id=p.patient_id, market=p.market.value,
    age=p.age, sex=p.sex.value, bmi=p.bmi,
    smoking_status=p.smoking_status.value,
    charlson_index=int(df.iloc[clean_idx]["charlson_index"]),
    n_diagnoses=len(p.diagnoses), n_labs=len(p.lab_results),
    risk_probability=proba,
    risk_class="Standard" if proba < 0.5 else "Substandard",
    concordance_index=cox.concordance_index,
    median_survival=15.0,
    dqs=dqs_r.dqs, dqs_tier=dqs_r.tier,
    dqs_completeness=dqs_r.completeness, dqs_consistency=dqs_r.consistency,
    dqs_recency=dqs_r.recency,
    validation_recommendation=val.recommendation,
    validation_explanation=val.explanation,
    flags_triggered=val.flags_triggered,
    top_features=[("age", 0.15), ("charlson_index", 0.12), ("bmi", 0.08),
                  ("has_hypertension", 0.07), ("smoking_current", 0.06)],
)

path = generate_report(data, "../data/reports/report_clean.pdf")
print(f"Clean report: {path}")
print(f"  Recommendation: {val.recommendation}")

Clean report: ../data/reports/report_clean.pdf
  Recommendation: accept


## Report 2: A flagged patient — plausible but wrong

In [5]:
p2 = cohort[flagged_idx]
dqs_r2 = dqs_results[flagged_idx]
proba2 = float(y_proba[flagged_idx])

val2 = detect_failure_modes(
    patient_id=p2.patient_id, dqs=dqs_r2.dqs, dqs_tier=dqs_r2.tier,
    raw_proba=proba2,
)

data2 = ReportData(
    patient_id=p2.patient_id, market=p2.market.value,
    age=p2.age, sex=p2.sex.value, bmi=p2.bmi,
    smoking_status=p2.smoking_status.value,
    charlson_index=int(df.iloc[flagged_idx]["charlson_index"]),
    n_diagnoses=len(p2.diagnoses), n_labs=len(p2.lab_results),
    risk_probability=proba2,
    risk_class="Standard" if proba2 < 0.5 else "Substandard",
    concordance_index=cox.concordance_index,
    median_survival=8.0,
    dqs=dqs_r2.dqs, dqs_tier=dqs_r2.tier,
    dqs_completeness=dqs_r2.completeness, dqs_consistency=dqs_r2.consistency,
    dqs_recency=dqs_r2.recency,
    validation_recommendation=val2.recommendation,
    validation_explanation=val2.explanation,
    flags_triggered=val2.flags_triggered,
    top_features=[("age", 0.18), ("bmi", 0.10), ("charlson_index", 0.09)],
)

path2 = generate_report(data2, "../data/reports/report_flagged.pdf")
print(f"Flagged report: {path2}")
print(f"  Recommendation: {val2.recommendation}")
print(f"  Explanation: {val2.explanation}")

Flagged report: ../data/reports/report_flagged.pdf
  Recommendation: reject_prediction
  Explanation: PLAUSIBLE-BUT-WRONG: Model confidence (100%) is not supported by data quality (DQS=0.58, tier=insufficient). The model cannot have enough information to justify this prediction. Escalate to human underwriter.


## Key finding

The clean report shows an **accept** recommendation — all validation checks passed. The flagged report shows a **reject_prediction** recommendation — the model's confidence is not supported by the data quality.

In a production workflow, the first patient proceeds through automated underwriting. The second patient is escalated to a human underwriter.

**This is the value proposition: the system doesn't just classify risk — it knows when to ask for help.**